# EEG Representational Similarity Analysis (RSA) Pipeline
## Replication: Wilson et al. — ERP CORE Dataset

This notebook implements a basic RSA pipeline for EEG data using the [ERP CORE dataset](https://erpinfo.org/erp-core). The pipeline covers:

1. Data loading and preprocessing (via MNE-Python)
2. Epoch extraction and baseline correction
3. Constructing neural Representational Dissimilarity Matrices (RDMs)
4. Constructing model RDMs (categorical and identity)
5. Spearman RSA — correlating neural and model RDMs
6. Time-resolved RSA across the epoch window
7. Visualization of RDMs and RSA time courses

**Dataset**: ERP CORE N170 component (face/object discrimination), Participant 1 used as demonstration.  
**Reference**: Kappenman et al. (2021), *NeuroImage*. https://doi.org/10.1016/j.neuroimage.2020.117465

---

## 0. Dependencies

In [ ]:
# Install dependencies if running in a fresh environment
# !pip install mne numpy scipy matplotlib seaborn requests tqdm

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from scipy.spatial.distance import squareform, pdist
import mne
from mne.datasets import erpcore
from tqdm import tqdm

mne.set_log_level('WARNING')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'MNE version: {mne.__version__}')
print('Dependencies loaded.')

## 1. Load ERP CORE Data (N170 — Face/Object)

The ERP CORE dataset is distributed through MNE-Python's dataset utilities. We use the N170 paradigm, which has participants viewing faces and objects — a classic paradigm for studying visual categorical representation.

Stimuli categories (from Kappenman et al., 2021):
- **Faces**: event codes 1–40
- **Cars**: event codes 41–80 (objects, non-face)
- **Scrambled variants**: 101–180 (excluded here)

We focus on faces vs. cars to probe categorical visual representation.

In [ ]:
# Download ERP CORE N170 data for participant 1
# MNE will cache the data locally on first run
SUBJECT_ID = 1

data_path = erpcore.data_path()  # downloads if not already cached
n170_path = data_path / f'sub-{SUBJECT_ID:03d}' / 'eeg'

# Load the raw EEG file (BrainVision format)
raw_fname = n170_path / f'sub-{SUBJECT_ID:03d}_task-N170_eeg.vhdr'
raw = mne.io.read_raw_brainvision(str(raw_fname), preload=True)

print(raw.info)
print(f'\nSampling frequency: {raw.info["sfreq"]} Hz')
print(f'Number of channels: {len(raw.ch_names)}')

## 2. Preprocessing

Standard EEG preprocessing steps:
- Band-pass filter (0.1–40 Hz)
- Re-reference to average reference
- Epoch extraction (−200 to 800 ms around stimulus onset)
- Baseline correction (−200 to 0 ms)

We retain face (event codes 1–40) and car (41–80) epochs, excluding scrambled stimuli.

In [ ]:
# --- Band-pass filtering ---
raw_filtered = raw.copy().filter(l_freq=0.1, h_freq=40.0, fir_window='hamming')

# --- Average reference ---
raw_filtered.set_eeg_reference('average', projection=True)
raw_filtered.apply_proj()

# --- Load events ---
events, event_id_dict = mne.events_from_annotations(raw_filtered)

# Build face (1-40) and car (41-80) event id mappings
face_ids = {str(i): i for i in range(1, 41)}
car_ids  = {str(i): i for i in range(41, 81)}
target_event_ids = {**face_ids, **car_ids}

# Keep only events that appear in this recording
available_ids = {k: v for k, v in target_event_ids.items() if v in events[:, 2]}

# --- Epoch extraction ---
TMIN, TMAX = -0.2, 0.8  # seconds
BASELINE   = (-0.2, 0.0)

epochs = mne.Epochs(
    raw_filtered,
    events,
    event_id=available_ids,
    tmin=TMIN,
    tmax=TMAX,
    baseline=BASELINE,
    preload=True,
    reject_by_annotation=True
)

print(epochs)

face_keys = [k for k in available_ids if int(k) <= 40]
car_keys  = [k for k in available_ids if int(k) >  40]

face_epochs = epochs[face_keys]
car_epochs  = epochs[car_keys]

print(f'\nFace epochs: {len(face_epochs)},  Car epochs: {len(car_epochs)}')
print(f'Epoch shape (n_epochs x n_channels x n_times): {face_epochs.get_data().shape}')

## 3. Construct Neural RDMs

For RSA, we construct a **Representational Dissimilarity Matrix (RDM)** from the EEG data.

**Approach — condition-averaged, pattern-based RDM at each time point:**
- Average trials within each stimulus condition to get a stable spatial pattern
- At each time point *t*, extract the pattern across channels for every condition
- Compute pairwise dissimilarity (1 − Pearson r) between condition patterns
- This yields a time-resolved RDM stack of shape (n_times × n_conditions × n_conditions)

In [ ]:
def compute_neural_rdm_timeresolved(epochs_obj, event_id_map):
    """
    Compute a time-resolved neural RDM.

    For each time point, compute pairwise (1 - Pearson r) dissimilarity
    across condition-averaged EEG spatial patterns.

    Parameters
    ----------
    epochs_obj   : mne.Epochs
    event_id_map : dict  {label_str: event_code}

    Returns
    -------
    rdms             : np.ndarray, shape (n_times, n_cond, n_cond)
    condition_labels : list of str
    times            : np.ndarray  (seconds)
    """
    keys   = sorted(event_id_map.keys(), key=lambda x: int(x))
    times  = epochs_obj.times
    n_times = len(times)
    n_cond  = len(keys)

    # Average over trials per condition -> shape (n_cond, n_ch, n_times)
    cond_data = np.stack([
        epochs_obj[k].get_data().mean(axis=0) for k in keys
    ], axis=0)

    rdms = np.zeros((n_times, n_cond, n_cond))
    for t_idx in range(n_times):
        patterns = cond_data[:, :, t_idx]          # (n_cond, n_ch)
        rdm_vec  = pdist(patterns, metric='correlation')  # 1 - r
        rdms[t_idx] = squareform(rdm_vec)

    return rdms, keys, times


# Use up to 10 face + 10 car conditions for a tractable demonstration
N_COND_EACH = 10

face_keys_sorted = sorted(face_keys, key=int)[:N_COND_EACH]
car_keys_sorted  = sorted(car_keys,  key=int)[:N_COND_EACH]

selected_keys   = face_keys_sorted + car_keys_sorted
selected_id_map = {k: available_ids[k] for k in selected_keys}

print(f'Computing time-resolved neural RDMs for {len(selected_keys)} conditions...')
neural_rdms, cond_labels, times = compute_neural_rdm_timeresolved(epochs, selected_id_map)

print(f'Neural RDM stack shape: {neural_rdms.shape}  (n_times, n_cond, n_cond)')
print(f'Time range: {times[0]:.3f} to {times[-1]:.3f} s  ({len(times)} time points)')

## 4. Construct Model RDMs

Model RDMs encode hypotheses about representational geometry.

**Models used:**

1. **Category model**: all face–car pairs are dissimilar (= 1); within-category pairs are similar (= 0).  
2. **Identity model**: all off-diagonal entries = 1 (uniform dissimilarity — null/baseline).

The Category model is our primary hypothesis: if the brain encodes face/car category distinctions at a given time point, the neural RDM should correlate with this model.

In [ ]:
n_cond  = len(selected_keys)
n_face  = N_COND_EACH
n_car   = N_COND_EACH

# Category model: 0 = same category, 1 = different category
category_labels = np.array([0] * n_face + [1] * n_car)
category_rdm    = (category_labels[:, None] != category_labels[None, :]).astype(float)

# Identity model: all stimuli maximally dissimilar (null)
identity_rdm = np.ones((n_cond, n_cond)) - np.eye(n_cond)

model_rdms = {
    'Category (Face vs. Car)': category_rdm,
    'Identity (Uniform Null)': identity_rdm,
}

print('Model RDMs constructed:')
for name, rdm in model_rdms.items():
    print(f'  {name}: shape={rdm.shape}, unique values={np.unique(rdm)}')

## 5. Time-Resolved RSA

At each time point, correlate the upper triangle of the neural RDM with each model RDM using **Spearman rank correlation**. This yields an RSA time course showing *when* the brain represents the structure predicted by each model.

The Spearman coefficient is preferred over Pearson here because:
- RDM values may not be normally distributed
- Rank-based correlation is more robust to outlier patterns

In [ ]:
def upper_tri_vec(rdm):
    """Extract upper triangle (excluding diagonal) as a flat vector."""
    idx = np.triu_indices(rdm.shape[0], k=1)
    return rdm[idx]


def time_resolved_rsa(neural_rdms, model_rdm):
    """
    Compute Spearman RSA at each time point.

    Parameters
    ----------
    neural_rdms : np.ndarray, shape (n_times, n_cond, n_cond)
    model_rdm   : np.ndarray, shape (n_cond, n_cond)

    Returns
    -------
    rsa_r : np.ndarray (n_times,)  Spearman r
    rsa_p : np.ndarray (n_times,)  p-values (uncorrected)
    """
    model_vec = upper_tri_vec(model_rdm)
    n_times   = neural_rdms.shape[0]
    rsa_r     = np.zeros(n_times)
    rsa_p     = np.zeros(n_times)
    for t in range(n_times):
        neural_vec    = upper_tri_vec(neural_rdms[t])
        r, p          = spearmanr(neural_vec, model_vec)
        rsa_r[t], rsa_p[t] = r, p
    return rsa_r, rsa_p


rsa_results = {}
for model_name, model_rdm in model_rdms.items():
    r, p = time_resolved_rsa(neural_rdms, model_rdm)
    rsa_results[model_name] = {'r': r, 'p': p}
    peak_idx = np.argmax(np.abs(r))
    print(f'{model_name}:')
    print(f'  Peak r = {r[peak_idx]:.4f}  at t = {times[peak_idx]*1000:.1f} ms  (p = {p[peak_idx]:.4f})')

## 6. Visualization

We produce three plots:
1. Model RDMs side-by-side
2. Neural RDM at the N170 peak (~170 ms post-stimulus)
3. RSA time courses with significance shading

In [ ]:
# --- Plot 1: Model RDMs ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Model Representational Dissimilarity Matrices', fontsize=13, fontweight='bold')

tick_labels = [f'F{i+1}' for i in range(n_face)] + [f'C{i+1}' for i in range(n_car)]

for ax, (name, rdm) in zip(axes, model_rdms.items()):
    im = ax.imshow(rdm, cmap='RdBu_r', vmin=0, vmax=1)
    ax.set_title(name, fontsize=11)
    ax.set_xticks(range(n_cond))
    ax.set_yticks(range(n_cond))
    ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
    ax.set_yticklabels(tick_labels, fontsize=7)
    ax.axhline(n_face - 0.5, color='k', lw=1.5, ls='--')
    ax.axvline(n_face - 0.5, color='k', lw=1.5, ls='--')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Dissimilarity')

plt.tight_layout()
plt.savefig('model_rdms.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_rdms.png')

In [ ]:
# --- Plot 2: Neural RDM at ~170 ms ---
target_s  = 0.170
t_idx_170 = np.argmin(np.abs(times - target_s))
actual_ms = times[t_idx_170] * 1000

fig, ax = plt.subplots(figsize=(6, 5))
rdm_170 = neural_rdms[t_idx_170]
vmax = np.percentile(rdm_170, 95)

im = ax.imshow(rdm_170, cmap='RdBu_r', vmin=0, vmax=vmax)
ax.set_title(f'Neural RDM at t ≈ {actual_ms:.0f} ms\n(1 − Pearson r)', fontsize=12)
ax.set_xticks(range(n_cond))
ax.set_yticks(range(n_cond))
ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
ax.set_yticklabels(tick_labels, fontsize=7)
ax.axhline(n_face - 0.5, color='k', lw=1.5, ls='--')
ax.axvline(n_face - 0.5, color='k', lw=1.5, ls='--')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Dissimilarity')
plt.tight_layout()
plt.savefig('neural_rdm_170ms.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: neural_rdm_170ms.png  (t = {actual_ms:.1f} ms)')

In [ ]:
# --- Plot 3: RSA time courses ---
ALPHA  = 0.05
colors = ['#E64B35', '#4DBBD5']

fig, ax = plt.subplots(figsize=(11, 4))

for (model_name, res), color in zip(rsa_results.items(), colors):
    r_vals = res['r']
    p_vals = res['p']
    t_ms   = times * 1000

    ax.plot(t_ms, r_vals, label=model_name, color=color, lw=2)
    sig_mask = p_vals < ALPHA
    ax.fill_between(t_ms, r_vals, 0, where=sig_mask,
                    alpha=0.25, color=color,
                    label=f'{model_name} significant (p<{ALPHA})')

ax.axhline(0,   color='k',    lw=0.8, ls='--')
ax.axvline(0,   color='gray', lw=0.8, ls=':',  label='Stimulus onset')
ax.axvline(170, color='gray', lw=0.8, ls='-.', alpha=0.5, label='N170 peak (~170 ms)')

ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Spearman r  (neural vs. model RDM)', fontsize=12)
ax.set_title('Time-Resolved RSA: EEG vs. Model RDMs', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim([times[0]*1000, times[-1]*1000])

sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('rsa_timecourse.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rsa_timecourse.png')

## 7. Summary & Interpretation

| Step | Output |
|------|--------|
| Preprocessing | Band-pass filtered, avg-referenced, baselined epochs |
| Neural RDMs | Time-resolved stack (n_times × n_cond × n_cond), correlation distance |
| Model RDMs | Category (face/car) and identity null |
| RSA | Spearman r time course with p < 0.05 significance shading |

**Expected findings** (per Wilson et al. and the broader N170 literature):
- The **Category model** RSA coefficient should peak around 150–200 ms post-stimulus, reflecting the well-known N170 sensitivity to face vs. object category.
- The **Identity model** (null) should remain near zero, confirming that categorical structure rather than uniform dissimilarity drives the neural signal.

---

### Potential Extensions
- **Permutation testing**: label-shuffled null distribution for robust inference
- **Model comparison**: add pixel-level RDM, DCNN layer RDMs, or behavioral similarity ratings
- **Group-level RSA**: repeat across all 40 ERP CORE participants and run second-level statistics
- **Temporal Generalization**: cross-time RSA matrix to assess representational stability
- **Searchlight RSA**: spatial RSA across electrode neighborhoods

In [ ]:
import sys, scipy
print(f'Python : {sys.version}')
print(f'NumPy  : {np.__version__}')
print(f'SciPy  : {scipy.__version__}')
print(f'MNE    : {mne.__version__}')